# TP 2 — métriques de régression

Mesurer un modèle de régression, puis regarder *comment* il se trompe.

## 1.0 Prédire la progression du diabète

`load_diabetes` donne 442 patients, 10 caractéristiques mesurées au départ (âge, sexe, IMC, tension, six analyses sanguines) et une cible $y$ : la progression de la maladie un an plus tard, de 25 à 346.

`model` est une régression linéaire entraînée sur 353 de ces patients ; les 89 autres forment le test, et ce sont ses prédictions $\hat y$ sur eux que nous mesurons.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

X, y = load_diabetes(return_X_y=True)
noise = np.random.default_rng(0).normal(size=(len(X), 342))   # pure-noise columns, added in 1.3
X_train, X_test, y_train, y_test, noise_train, noise_test = train_test_split(
    X, y, noise, test_size=0.2, random_state=0)
model = LinearRegression().fit(X_train, y_train)
y_hat = model.predict(X_test)

## 1.1 Quatre résumés des mêmes résidus

Le résidu du patient $i$ est son erreur, $e^{(i)} = y^{(i)} - \hat y^{(i)}$. Quatre métriques résument les $m$ résidus en un seul chiffre :

$$\text{MSE} = \frac{1}{m}\sum_{i=1}^{m}\big(\hat y^{(i)} - y^{(i)}\big)^2, \qquad \text{RMSE} = \sqrt{\text{MSE}}, \qquad \text{MAE} = \frac{1}{m}\sum_{i=1}^{m}\big|\hat y^{(i)} - y^{(i)}\big|,$$

$$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}, \qquad \text{SS}_{\text{res}} = \sum_{i=1}^{m}\big(y^{(i)} - \hat y^{(i)}\big)^2, \qquad \text{SS}_{\text{tot}} = \sum_{i=1}^{m}\big(y^{(i)} - \bar y\big)^2.$$

Écrivez-les avec numpy, sans boucle, à partir des seuls arguments `y` et `y_hat`.

**Q1.** Les résidus se comptent en points de progression, sur une échelle qui va de 25 à 346. Sans rien exécuter : dans quelle unité s'exprime chacune des quatre métriques, et laquelle du RMSE ou de la MAE sera la plus grande ? Et la ligne de base « prédire la moyenne du train » : quel R² attendez-vous d'elle ?

**Votre réponse :**



In [ ]:
def mse(y, y_hat):
    # TODO: compute the mean of the squared errors
    ...


def rmse(y, y_hat):
    # TODO: compute the square root of the MSE
    ...


def mae(y, y_hat):
    # TODO: compute the mean of the absolute errors
    ...


def r2(y, y_hat):
    """1 - SS_res / SS_tot, where SS_tot uses the mean of the true values y."""
    # TODO: compute ss_res, ss_tot, then return 1 - ss_res / ss_tot
    ...


assert np.isclose(mse(y_test, y_hat), mean_squared_error(y_test, y_hat))
assert np.isclose(rmse(y_test, y_hat), np.sqrt(mean_squared_error(y_test, y_hat)))
assert np.isclose(mae(y_test, y_hat), mean_absolute_error(y_test, y_hat))
assert np.isclose(r2(y_test, y_hat), r2_score(y_test, y_hat))
# same functions on the training set: they must read their arguments, not y_test
y_hat_train = model.predict(X_train)
assert np.isclose(mse(y_train, y_hat_train), mean_squared_error(y_train, y_hat_train))
assert np.isclose(rmse(y_train, y_hat_train), np.sqrt(mean_squared_error(y_train, y_hat_train)))
assert np.isclose(mae(y_train, y_hat_train), mean_absolute_error(y_train, y_hat_train))
assert np.isclose(r2(y_train, y_hat_train), r2_score(y_train, y_hat_train))

y_baseline = np.full(len(y_test), y_train.mean())   # a "model" that ignores x and always predicts the train mean

print(f"moyenne de y : {y_train.mean():.1f} sur le train, {y_test.mean():.1f} sur le test")
pd.DataFrame([[f(y_test, p) for f in (mse, rmse, mae, r2)] for p in (y_hat, y_baseline)],
             index=["modèle", "moyenne du train"],
             columns=["MSE", "RMSE", "MAE", "R²"]).round({"MSE": 1, "RMSE": 2, "MAE": 2, "R²": 4})

**Q2.** Traduisez le RMSE en une phrase pour un médecin. La ligne de base obtient-elle exactement $R^2 = 0$ ? Chiffrez l'écart, et dites ce qu'il faudrait changer à cette ligne de base pour qu'elle l'obtienne exactement. Et que dit le R² du modèle sur ce qu'il a appris ?

**Votre réponse :**



## 1.2 Un zéro de trop à la saisie

Le patient 0 du test a une progression de 321 ; à la saisie, quelqu'un a tapé 3210. Le modèle, lui, ne change pas : sa prédiction pour ce patient est la même qu'avant, mais son erreur sur lui grandit de 2 889 points.

**Q3.** Par combien attendez-vous que le RMSE et la MAE soient multipliés — par le même facteur ?

**Votre réponse :**



In [ ]:
y_test_typo = y_test.copy()
y_test_typo[0] = 3210            # one extra zero at data entry; the true value is 321

metrics = (mse, rmse, mae, r2)
before = [f(y_test, y_hat) for f in metrics]
# TODO: the same four metrics, with y_test_typo instead of y_test
after = ...
assert np.allclose(after[1:3], [np.sqrt(mean_squared_error(y_test_typo, y_hat)),
                                mean_absolute_error(y_test_typo, y_hat)])

table = pd.DataFrame({"avant": before, "après": after}, index=["MSE", "RMSE", "MAE", "R²"])
table["facteur"] = table["après"] / table["avant"]
table.loc["R²", "facteur"] = np.nan          # a ratio of two R² would mean nothing

print(f"ce seul patient (ŷ = {y_hat[0]:.1f}) pèse "
      f"{100 * (y_test_typo[0] - y_hat[0]) ** 2 / np.sum((y_test_typo - y_hat) ** 2):.1f} %"
      " de la somme des carrés des erreurs")
table.round(2)

**Q4.** Le R² tombe de 0,33 à 0,06 alors que le modèle n'a pas bougé : pourquoi une métrique sans unité, censée se comparer d'un problème à l'autre, encaisse-t-elle autant — que deviennent $\text{SS}_{\text{res}}$ *et* $\text{SS}_{\text{tot}}$, et pourquoi le tableau refuse-t-il de lui donner un facteur ? Quelle métrique rapporter si les données peuvent contenir des erreurs de saisie, et que faut-il faire des données elles-mêmes ?

**Votre réponse :**



## 1.3 Les deux visages du R²

$R^2 = 0$, c'est « aussi bien que prédire $\bar y$ » ; $R^2 < 0$, c'est pire. Deux pièges guettent celui qui lit un R².

### (a) Un modèle qui mémorise

`tree` est un arbre de décision sans profondeur maximale.

### (b) Des caractéristiques inutiles

On ajoute aux 10 vraies caractéristiques les $k$ premières colonnes de `noise` — du pur hasard, sans lien avec le diabète — on entraîne une régression linéaire comme `model`, et on mesure son R² sur l'entraînement et sur le test.

**Q5.** Pour chacun des deux modèles, à quoi vous attendez-vous côté entraînement, et côté test ? Une colonne de hasard en plus peut-elle faire *baisser* le R² d'entraînement ?

**Votre réponse :**



In [ ]:
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)

print(f"R² de l'arbre : entraînement {r2(y_train, tree.predict(X_train)):.3f}, "
      f"test {r2(y_test, tree.predict(X_test)):.3f}")
print(f"RMSE de test : arbre {rmse(y_test, tree.predict(X_test)):.1f}, "
      f"moyenne du train {rmse(y_test, y_baseline):.1f}")

In [ ]:
ks = [0, 10, 20, 50, 100, 150, 200, 250, 300]
r2_train, r2_test = [], []
for k in ks:
    Xk_train = np.hstack([X_train, noise_train[:, :k]])
    Xk_test = np.hstack([X_test, noise_test[:, :k]])
    # TODO: fit LinearRegression on (Xk_train, y_train), then append its R2 on the train set
    # to r2_train and its R2 on the test set to r2_test
    ...

assert np.isclose(r2_train[0], r2_score(y_train, model.predict(X_train)))   # k = 0 is the model of 1.0
assert np.isclose(r2_test[0], r2_score(y_test, model.predict(X_test)))

print(pd.DataFrame({"R² entraînement": r2_train, "R² test": r2_test}, index=ks).T.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ks, r2_train, "o-", label="R² d'entraînement")
ax.plot(ks, r2_test, "o-", label="R² de test")
ax.axhline(0, color="gray", ls="--", lw=1)
ax.annotate("R² de test = " + f"{r2_test[-1]:.2f}".replace(".", ",").replace("-", "−") + "\n(hors cadre)",
            xy=(ks[-1], -1.5), xytext=(-12, 30), textcoords="offset points", ha="right", fontsize=9,
            arrowprops=dict(arrowstyle="->"))
ax.set_ylim(-1.5, 1.1)
ax.set_xlabel("nombre k de colonnes de bruit pur ajoutées aux 10 caractéristiques")
ax.set_ylabel("R²")
ax.set_title("R² d'entraînement et de test selon le nombre de colonnes de bruit")
ax.legend(loc="lower left")
plt.show()

**Q6.** Le R² de test de l'arbre est négatif : qu'est-ce que cela veut dire concrètement, RMSE à l'appui ? À partir de combien de colonnes de bruit la régression fait-elle pire que la moyenne sur le test ? Sur les deux courbes, à quoi voit-on le **surapprentissage** — un modèle qui épouse le bruit de son entraînement au point de se dégrader ailleurs ?

**Votre réponse :**



## 1.4 Même R², trois diagnostics

Trois jeux mystères, une caractéristique $x$ chacun, 200 points d'entraînement et 100 de test. Les trois R² de test sont égaux à 0,01 près. On lit le diagnostic sur la forme du nuage des résidus : **nuage aléatoire** sans structure, **cône** (les résidus s'écartent quand $\hat y$ grandit : c'est l'**hétéroscédasticité**), ou **U** (résidus positifs aux deux bouts, négatifs au milieu).

**Q7.** Avant de voir les figures : laquelle des trois formes est la seule à dire « ce modèle est sain partout », et que trahit chacune des deux autres ?

**Votre réponse :**



In [ ]:
#@title Jeux mystères — exécutez sans lire le code, il donne la réponse
datasets = {}
rng = np.random.default_rng(1); x = rng.uniform(0, 10, 300); datasets["jeu 1"] = (x, 3 * x + 5 + rng.normal(0, 1, 300) * 0.66 * x)
rng = np.random.default_rng(2); x = rng.uniform(0, 10, 300); datasets["jeu 2"] = (x, 0.5 * (x - 2) ** 2 + 5 + rng.normal(0, 2.2, 300))
rng = np.random.default_rng(3); x = rng.uniform(0, 10, 300); datasets["jeu 3"] = (x, 3 * x + 5 + rng.normal(0, 3.74, 300))

results = {}      # dataset name -> (test targets, test predictions)
for name, (x, y_set) in datasets.items():
    x_tr, x_te, y_tr, y_te = train_test_split(x.reshape(-1, 1), y_set, test_size=1 / 3, random_state=0)
    results[name] = (y_te, LinearRegression().fit(x_tr, y_tr).predict(x_te))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, (y_set, y_set_hat)) in zip(axes, results.items()):
    # TODO: compute the residuals of this dataset, e = y - y_hat
    residuals = ...
    ax.scatter(y_set_hat, residuals, s=18, alpha=0.8)
    ax.axhline(0, color="gray", lw=1.5)
    ax.set_title(f"{name} (R² de test = " + f"{r2_score(y_set, y_set_hat):.2f}".replace(".", ",") + ")")
    ax.set_xlabel(r"prédiction $\hat{y}$")
    ax.set_ylabel(r"résidu $e = y - \hat{y}$")
plt.tight_layout()
plt.show()

**Q8.** Associez chaque jeu à son diagnostic. Pour le jeu en cône, où les prédictions sont-elles fiables ? Pour le jeu en U, que proposeriez-vous ?

**Votre réponse :**

